In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.pipeline import DATA, RESULTS

z = np.load(DATA / "windows" / "tune.npz", allow_pickle=True)
X, V, ids = z["X"], z["mask"], z["win_ids"]
typ = z["attack_type"].astype(str)

m4 = pd.read_parquet(RESULTS / "scores" / "m4_tcn_nftoniot_seed11.parquet")
m4 = m4[m4["split"] == "tune"].set_index("id")["score"]
score = m4.reindex(ids).to_numpy()                 # align scores to the window order

full = V.all(1)                                    # windows with all 16 flows present
disp = X[full].std(axis=1).mean(axis=1)            # within-window spread, averaged over features

df = pd.DataFrame({"type": typ[full], "dispersion": disp, "m4_score": score[full]})
print(df.groupby("type")[["dispersion", "m4_score"]].median().round(3))
print("\nSpearman(dispersion, m4_score) over all full windows:",
      round(spearmanr(df["dispersion"], df["m4_score"])[0], 3))
			

           dispersion  m4_score
type                           
Benign          0.311     0.304
ddos            0.991     1.413
dos             1.060     1.672
injection       1.010     1.751
mitm            0.849     0.945
password        1.210     2.266
scanning        0.071     0.178
xss             1.400     2.686

Spearman(dispersion, m4_score) over all full windows: 0.927
